# Identify disease populations

In [ ]:
# import packages
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad
import os

In [ ]:
# Set figure params to now show frame on umaps
sc.settings.set_figure_params(dpi=120)
sc.set_figure_params(figsize=(8, 8), transparent=True)

# For random shuffling of cells before plotting
rng = np.random.default_rng(0)

In [ ]:
sc.settings.figdir = 'images/'


In [ ]:
os.getcwd()

In [ ]:
os.chdir('SET YOUR WORKING DIRECTORY HERE')

In [ ]:
# load data

lin_list =['myeloid', 'epithelial', 'lymphoid', 'stroma']

In [ ]:
adata_myeloid_full = sc.read_h5ad("data/taurus_data/taurus_hgca_"+lin_list[0]+"_hgca_concat.h5ad")
adata_epi_full = sc.read_h5ad("data/taurus_data/taurus_hgca_"+lin_list[1]+"_hgca_concat.h5ad")
adata_lymph_full = sc.read_h5ad("data/taurus_data/taurus_hgca_"+lin_list[2]+"_hgca_concat.h5ad")
adata_stromal_full = sc.read_h5ad("data/taurus_data/taurus_hgca_"+lin_list[3]+"_hgca_concat.h5ad")


In [ ]:
adata_myeloid_full

In [ ]:
ad_concat = ad.concat([adata_myeloid_full, 
                       adata_epi_full, 
                       adata_lymph_full, 
                       adata_stromal_full])

In [ ]:
ad_concat = ad_concat[ad_concat.obs['subset']=='Query']

In [ ]:
df =pd.crosstab(ad_concat.obs['author_cell_type'], ad_concat.obs['hgca_celltype_v1'], margins=True, margins_name='Total') # Incl. margins to get the total number of cells per label

In [ ]:
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec

row_tot = df.loc[:, 'Total'].drop('Total')      # cells per final_analysis label
col_tot = df.loc['Total', :].drop('Total')      # cells per prediction label
mat     = df.drop(index='Total', columns='Total')

NORM = 'row'          # 'row' -> % of each true label; 'col' -> % of each prediction
norm = (mat.div(mat.sum(1), axis=0) if NORM == 'row'
        else mat.div(mat.sum(0), axis=1)) * 100

# optional: order columns so the dominant mapping runs along the diagonal
order = norm.idxmax(1).map({c: i for i, c in enumerate(norm.columns)}).sort_values().index
norm, mat, row_tot = norm.loc[order], mat.loc[order], row_tot.loc[order]


n_row, n_col = norm.shape


In [ ]:
def plot_agreement_heatmap(df, NORM='row'):

    # ---------------------------------------------------------------- layout
    fig = plt.figure(figsize=(0.17 * n_col + 4.5, 0.17 * n_row + 4.5))
    gs = GridSpec(2, 3, figure=fig,
                width_ratios=[n_col, 9, 1.2], height_ratios=[9, n_row],
                wspace=0.02, hspace=0.02)

    ax_t  = fig.add_subplot(gs[0, 0])                  # column totals, on top
    ax_hm = fig.add_subplot(gs[1, 0], sharex=ax_t)     # heatmap
    ax_r  = fig.add_subplot(gs[1, 1], sharey=ax_hm)    # row totals
    ax_cb = fig.add_subplot(gs[1, 2])

    # ---------------------------------------------------------------- heatmap
    cmap = plt.get_cmap('Blues').copy()
    cmap.set_bad('#f5f5f5')                      # zeros read as empty, not "low"
    data = np.ma.masked_where(norm.values == 0, norm.values)

    mesh = ax_hm.pcolormesh(data, cmap=cmap, vmin=0, vmax=100,
                            edgecolors='white', linewidth=0.3)

    ax_hm.set_xlim(0, n_col); ax_hm.set_ylim(n_row, 0)
    ax_hm.set_yticks(np.arange(n_row) + .5, norm.index, fontsize=6)
    ax_hm.set_xticks(np.arange(n_col) + .5, norm.columns, rotation=90, fontsize=6)
    ax_hm.set_ylabel('author_cell_type', fontsize=10)
    ax_hm.set_xlabel('hgca_celltype_v1', fontsize=10)
    for s in ax_hm.spines.values():
        s.set_visible(False)

    cb = fig.colorbar(mesh, cax=ax_cb)
    cb.set_label(f'% of {NORM}', fontsize=9)
    cb.ax.tick_params(labelsize=7)
    cb.outline.set_visible(False)

    # ---------------------------------------------------------------- top bars
    ax_t.bar(np.arange(n_col) + .5, col_tot.values, width=0.8,
            color='#94a3b8', linewidth=0)
    ax_t.set_yscale('log')
    ax_t.set_ylabel('n cells', fontsize=9)
    ax_t.tick_params(axis='x', bottom=False, labelbottom=False)   # labels live on the heatmap
    ax_t.tick_params(axis='y', labelsize=7)
    ax_t.grid(axis='y', color='white', lw=0.8)
    ax_t.set_axisbelow(True)
    for s in ax_t.spines.values():
        s.set_visible(False)

    # ---------------------------------------------------------------- right bars
    ax_r.barh(np.arange(n_row) + .5, row_tot.values, height=0.8,
            color='#94a3b8', linewidth=0)
    ax_r.set_xscale('log')
    ax_r.set_xlabel('n cells', fontsize=9)
    ax_r.tick_params(axis='y', left=False, labelleft=False)
    ax_r.tick_params(axis='x', labelsize=7)
    ax_r.grid(axis='x', color='white', lw=0.8)
    ax_r.set_axisbelow(True)
    for s in ax_r.spines.values():
        s.set_visible(False)

    fig.suptitle(f'Label transfer: author_cell_type vs hgca_celltype_v1  (row-normalised, n={mat.values.sum():,})',
                fontsize=11, y=0.995)
    #fig.savefig('label_transfer_heatmap.png', dpi=200, bbox_inches='tight')
    plt.show()

In [ ]:
plot_agreement_heatmap(df)

In [ ]:
adata = ad_concat.copy()

In [ ]:
adata.X[:100].toarray()[adata.X[:100].toarray()>0]

In [ ]:
sc.pp.normalize_total(adata, target_sum=10e4)
sc.pp.log1p(adata)

In [ ]:
sc.pp.neighbors(adata_myeloid_full, use_rep='X_scANVI_'+str(lin_list[0])+'_lin')
sc.tl.umap(adata_myeloid_full)

In [ ]:
custom_colors = {
    'Reference': '#1f77b4', 
    'Query': "#ff320e"
}

In [ ]:
sc.pl.umap(
    adata_myeloid_full[rng.permutation(adata_myeloid_full.n_obs)],
    color=['subset'],
    ncols=2,
    wspace=0.5,
    s=5,
    frameon=False,
    palette=custom_colors,
    save=f'_{lin_list[0]}.svg')

In [ ]:
adata_myeloid_full

In [ ]:
sc.pl.umap(
    adata_epi_full[rng.permutation(adata_epi_full.n_obs)],
    color=['subset'],
    ncols=2,
    wspace=0.5,
    s=2,
    frameon=False,
    palette=custom_colors,
    alpha=0.7,
    save=f'_{lin_list[1]}.svg')

In [ ]:
sc.pl.umap(
    adata_lymph_full[rng.permutation(adata_lymph_full.n_obs)],
    color=['subset'],
    ncols=2,
    wspace=0.5,
    s=2,
    frameon=False,
    alpha=0.7,
    palette=custom_colors,
    save=f'_{lin_list[2]}.svg')

In [ ]:
sc.pl.umap(
    adata_stromal_full[rng.permutation(adata_stromal_full.n_obs)],
    color=['subset'],
    ncols=2,
    wspace=0.5,
    s=3,
    frameon=False,
    alpha=0.8,
    palette=custom_colors,
    save=f'_{lin_list[3]}.svg')

In [ ]:
adata_myeloid_full

# Milo on inflammation in general

In [ ]:
## Initialize object for Milo analysis

lin_idx = 0

lin_adata = taurus_adata_list[lin_idx]
lin_adata = lin_adata[lin_adata.obs['Ileum_vs_Colon']=='Colon'].copy()
lin_adata.obs["Inflammation_binary"] = lin_adata.obs["Inflammation"].replace({"Healthy":"Non_Inflamed"})

In [ ]:
milo = pt.tl.Milo()
mdata = milo.load(lin_adata)

In [ ]:
sc.pp.neighbors(mdata["rna"], use_rep="X_scANVI_myeloid_lin", n_neighbors=50)

In [ ]:
milo.make_nhoods(mdata["rna"], prop=0.1)

In [ ]:
mdata = milo.count_nhoods(mdata, sample_col="sample_id")

In [ ]:
mdata

In [ ]:
mdata["rna"].obs["Inflammation_binary"].value_counts()

In [ ]:
mdata["rna"].obs["Inflammation_binary"] = mdata["rna"].obs["Inflammation_binary"].cat.reorder_categories(["Non_Inflamed", "Inflamed"])

In [ ]:
milo.da_nhoods(mdata, design="~Inflammation_score", solver="edger")

In [ ]:
milo.build_nhood_graph(mdata)

In [ ]:
plt.rcParams["figure.figsize"] = [7, 7]
milo.plot_nhood_graph(
    mdata,
    padj_threshold=0.1,  # SpatialFDR level (1%)
    min_size=1,  # Size of smallest dot
)

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='hgca_celltype_v1')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Inflammation_score')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Remission_status')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Patient')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Treatment')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='final_analysis', legend_loc="on data", legend_fontsize=8)

### HGCA cell types

In [ ]:
milo.annotate_nhoods(mdata, anno_col="hgca_celltype_v1")

In [ ]:
milo.plot_da_beeswarm(mdata, padj_threshold=0.1)

### On author labels

In [ ]:
milo.annotate_nhoods(mdata, anno_col="final_analysis")

In [ ]:
milo.plot_da_beeswarm(mdata, padj_threshold=0.1)

In [ ]:
adata = lin_adata

In [ ]:
sc.pp.normalize_total(adata, target_sum=10e4)
sc.pp.log1p(adata)

In [ ]:
import numpy as np, pandas as pd, scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt, seaborn as sns

L1, L2 = "hgca_celltype_v1", "final_analysis"

# pseudobulk correlation assumes log-normalized values, NOT raw counts
print(adata.X.min(), adata.X.max(), adata.X.dtype)   # max should be ~5-10, not thousands

In [ ]:
X = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
expr = pd.DataFrame(X, index=adata.obs_names, columns=adata.var_names)

def pseudobulk(expr, labels, min_cells=10):
    lab = pd.Series(np.asarray(labels).astype(str), index=expr.index)
    keep = lab.value_counts()[lambda s: s >= min_cells].index
    pb = expr[lab.isin(keep).values].groupby(lab[lab.isin(keep)].values).mean()
    return pb

pb1 = pseudobulk(expr, adata.obs[L1])   # clusters x genes
pb2 = pseudobulk(expr, adata.obs[L2])

# z-score each gene across the *union* of both label sets so the two
# matrices sit in the same space and highly-expressed genes don't dominate
both = pd.concat([pb1, pb2])
z = (both - both.mean(0)) / both.std(0).replace(0, np.nan)
z = z.dropna(axis=1)
z1, z2 = z.iloc[:len(pb1)], z.iloc[len(pb1):]

# Pearson correlation between every hgca cluster and every final_analysis cluster
sim = pd.DataFrame(np.corrcoef(z1.values, z2.values)[:len(z1), len(z1):],
                   index=z1.index, columns=z2.index)

g = sns.clustermap(sim, cmap="RdBu_r", center=0, figsize=(12, 10),
                   cbar_kws={"label": "Pearson r (pseudobulk, gene-z)"},
                   annot=False, linewidths=.2)
g.ax_heatmap.set_xlabel(L2); g.ax_heatmap.set_ylabel(L1)
plt.show()

## Milo on Epithelial Cells

In [ ]:
## Initialize object for Milo analysis

lin_idx = 1

lin_adata = taurus_adata_list[lin_idx]
#lin_adata = lin_adata[[s in ['Non_Remission', 'Remission'] for s in lin_adata.obs['Remission_status']]]
#lin_adata = lin_adata[lin_adata.obs['Treatment']=='Pre'].copy()
#lin_adata = lin_adata[lin_adata.obs['Disease']=='CD'].copy()

lin_adata = lin_adata[lin_adata.obs['Ileum_vs_Colon']=='Colon'].copy()

lin_adata.obs["Inflammation_binary"] = lin_adata.obs["Inflammation"].replace({"Healthy":"Non_Inflamed"})

In [ ]:
milo = pt.tl.Milo()
mdata_epi = milo.load(lin_adata)

In [ ]:
mdata_epi

In [ ]:
sc.pp.neighbors(mdata_epi["rna"], use_rep=f"X_scANVI_{lin_list[lin_idx]}_lin", n_neighbors=100)

In [ ]:
milo.make_nhoods(mdata_epi["rna"], prop=0.1)

In [ ]:
mdata_epi = milo.count_nhoods(mdata_epi, sample_col="sample_id")

In [ ]:
mdata_epi

In [ ]:
mdata_epi["rna"].obs["Inflammation_binary"].value_counts()

In [ ]:
mdata_epi["rna"].obs["Inflammation_binary"] = mdata_epi["rna"].obs["Inflammation_binary"].cat.reorder_categories(["Non_Inflamed", "Inflamed"])

In [ ]:
milo.da_nhoods(mdata_epi, design="~Inflammation_score", solver="edger")

In [ ]:
milo.build_nhood_graph(mdata_epi)

In [ ]:
plt.rcParams["figure.figsize"] = [7, 7]
milo.plot_nhood_graph(
    mdata_epi,
    padj_threshold=0.1,  # SpatialFDR level (1%)
    min_size=1,  # Size of smallest dot
)

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='hgca_celltype_v1')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Inflammation_score')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Remission_status')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Patient')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Treatment')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='final_analysis', legend_loc="on data", legend_fontsize=8)

### HGCA cell types

In [ ]:
milo.annotate_nhoods(mdata_epi, anno_col="hgca_celltype_v1")

In [ ]:
milo.plot_da_beeswarm(mdata_epi, padj_threshold=0.1)

### On author labels

In [ ]:
milo.annotate_nhoods(mdata_epi, anno_col="final_analysis")

In [ ]:
milo.plot_da_beeswarm(mdata_epi, padj_threshold=0.1)

In [ ]:
adata = lin_adata

In [ ]:
sc.pp.normalize_total(adata, target_sum=10e4)
sc.pp.log1p(adata)

In [ ]:
import numpy as np, pandas as pd, scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt, seaborn as sns

L1, L2 = "hgca_celltype_v1", "final_analysis"

# pseudobulk correlation assumes log-normalized values, NOT raw counts
print(adata.X.min(), adata.X.max(), adata.X.dtype)   # max should be ~5-10, not thousands

In [ ]:
X = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
expr = pd.DataFrame(X, index=adata.obs_names, columns=adata.var_names)

def pseudobulk(expr, labels, min_cells=10):
    lab = pd.Series(np.asarray(labels).astype(str), index=expr.index)
    keep = lab.value_counts()[lambda s: s >= min_cells].index
    pb = expr[lab.isin(keep).values].groupby(lab[lab.isin(keep)].values).mean()
    return pb

pb1 = pseudobulk(expr, adata.obs[L1])   # clusters x genes
pb2 = pseudobulk(expr, adata.obs[L2])

# z-score each gene across the *union* of both label sets so the two
# matrices sit in the same space and highly-expressed genes don't dominate
both = pd.concat([pb1, pb2])
z = (both - both.mean(0)) / both.std(0).replace(0, np.nan)
z = z.dropna(axis=1)
z1, z2 = z.iloc[:len(pb1)], z.iloc[len(pb1):]

# Pearson correlation between every hgca cluster and every final_analysis cluster
sim = pd.DataFrame(np.corrcoef(z1.values, z2.values)[:len(z1), len(z1):],
                   index=z1.index, columns=z2.index)

g = sns.clustermap(sim, cmap="RdBu_r", center=0, figsize=(12, 10),
                   cbar_kws={"label": "Pearson r (pseudobulk, gene-z)"},
                   annot=False, linewidths=.2)
g.ax_heatmap.set_xlabel(L2); g.ax_heatmap.set_ylabel(L1)
plt.show()

## Milo on Lymphoid Cells

In [ ]:
## Initialize object for Milo analysis

lin_idx = 2

lin_adata = taurus_adata_list[lin_idx]
#lin_adata = lin_adata[[s in ['Non_Remission', 'Remission'] for s in lin_adata.obs['Remission_status']]]
#lin_adata = lin_adata[lin_adata.obs['Treatment']=='Pre'].copy()
#lin_adata = lin_adata[lin_adata.obs['Disease']=='CD'].copy()

lin_adata = lin_adata[lin_adata.obs['Ileum_vs_Colon']=='Colon'].copy()

lin_adata.obs["Inflammation_binary"] = lin_adata.obs["Inflammation"].replace({"Healthy":"Non_Inflamed"})

In [ ]:
milo = pt.tl.Milo()
mdata_lymph = milo.load(lin_adata)

In [ ]:
mdata_lymph

In [ ]:
sc.pp.neighbors(mdata_lymph["rna"], use_rep=f"X_scANVI_{lin_list[lin_idx]}_lin", n_neighbors=100)

In [ ]:
milo.make_nhoods(mdata_lymph["rna"], prop=0.1)

In [ ]:
mdata_lymph = milo.count_nhoods(mdata_lymph, sample_col="sample_id")

In [ ]:
mdata_lymph

In [ ]:
mdata_lymph["rna"].obs["Inflammation_binary"].value_counts()

In [ ]:
mdata_lymph["rna"].obs["Inflammation_binary"] = mdata_lymph["rna"].obs["Inflammation_binary"].cat.reorder_categories(["Non_Inflamed", "Inflamed"])

In [ ]:
milo.da_nhoods(mdata_lymph, design="~Inflammation_score", solver="edger")

In [ ]:
milo.build_nhood_graph(mdata_lymph)

In [ ]:
plt.rcParams["figure.figsize"] = [7, 7]
milo.plot_nhood_graph(
    mdata_lymph,
    padj_threshold=0.1,  # SpatialFDR level (1%)
    min_size=1,  # Size of smallest dot
)

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='hgca_celltype_v1')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Inflammation_score')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Remission_status')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Patient')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Treatment')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='final_analysis', legend_loc="on data", legend_fontsize=8)

### HGCA cell types

In [ ]:
milo.annotate_nhoods(mdata_lymph, anno_col="hgca_celltype_v1")

In [ ]:
sc.settings.figdir = 'hca-gut-atlas-tutorial/images/temp_images/'
sc.settings.set_figure_params(dpi=180, format='svg', transparent=True)

In [ ]:
lin_idx=2
plt.rcParams["figure.figsize"] = (4, 8)

fig =milo.plot_da_beeswarm(
    mdata_lymph,
    padj_threshold=0.1,
    return_fig=True
    )

fig.savefig(
    f'hca-gut-atlas-tutorial/images/temp_images/milo_beeswarm_{lin_list[lin_idx]}.svg',
    format='svg',
    bbox_inches='tight',
    transparent=True)


### On author labels

In [ ]:
milo.annotate_nhoods(mdata_lymph, anno_col="final_analysis")

In [ ]:
milo.plot_da_beeswarm(mdata_lymph, padj_threshold=0.1)

In [ ]:
adata = lin_adata

In [ ]:
sc.pp.normalize_total(adata, target_sum=10e4)
sc.pp.log1p(adata)

In [ ]:
import numpy as np, pandas as pd, scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt, seaborn as sns

L1, L2 = "hgca_celltype_v1", "final_analysis"

# pseudobulk correlation assumes log-normalized values, NOT raw counts
print(adata.X.min(), adata.X.max(), adata.X.dtype)   # max should be ~5-10, not thousands

In [ ]:
X = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
expr = pd.DataFrame(X, index=adata.obs_names, columns=adata.var_names)

def pseudobulk(expr, labels, min_cells=10):
    lab = pd.Series(np.asarray(labels).astype(str), index=expr.index)
    keep = lab.value_counts()[lambda s: s >= min_cells].index
    pb = expr[lab.isin(keep).values].groupby(lab[lab.isin(keep)].values).mean()
    return pb

pb1 = pseudobulk(expr, adata.obs[L1])   # clusters x genes
pb2 = pseudobulk(expr, adata.obs[L2])

# z-score each gene across the *union* of both label sets so the two
# matrices sit in the same space and highly-expressed genes don't dominate
both = pd.concat([pb1, pb2])
z = (both - both.mean(0)) / both.std(0).replace(0, np.nan)
z = z.dropna(axis=1)
z1, z2 = z.iloc[:len(pb1)], z.iloc[len(pb1):]

# Pearson correlation between every hgca cluster and every final_analysis cluster
sim = pd.DataFrame(np.corrcoef(z1.values, z2.values)[:len(z1), len(z1):],
                   index=z1.index, columns=z2.index)

g = sns.clustermap(sim, cmap="RdBu_r", center=0, figsize=(12, 10),
                   cbar_kws={"label": "Pearson r (pseudobulk, gene-z)"},
                   annot=False, linewidths=.2)
g.ax_heatmap.set_xlabel(L2); g.ax_heatmap.set_ylabel(L1)
plt.show()

In [ ]:
## Initialize object for Milo analysis

lin_idx = 3

lin_adata = taurus_adata_list[lin_idx]
#lin_adata = lin_adata[[s in ['Non_Remission', 'Remission'] for s in lin_adata.obs['Remission_status']]]
#lin_adata = lin_adata[lin_adata.obs['Treatment']=='Pre'].copy()
#lin_adata = lin_adata[lin_adata.obs['Disease']=='CD'].copy()

lin_adata = lin_adata[lin_adata.obs['Ileum_vs_Colon']=='Colon'].copy()

lin_adata.obs["Inflammation_binary"] = lin_adata.obs["Inflammation"].replace({"Healthy":"Non_Inflamed"})

In [ ]:
milo = pt.tl.Milo()
mdata_stroma = milo.load(lin_adata)

In [ ]:
mdata_stroma

In [ ]:
sc.pp.neighbors(mdata_stroma["rna"], use_rep=f"X_scANVI_{lin_list[lin_idx]}_lin", n_neighbors=100)

In [ ]:
milo.make_nhoods(mdata_stroma["rna"], prop=0.1)

In [ ]:
mdata_stroma = milo.count_nhoods(mdata_stroma, sample_col="sample_id")

In [ ]:
mdata_stroma

In [ ]:
mdata_stroma["rna"].obs["Inflammation_binary"].value_counts()

In [ ]:
mdata_stroma["rna"].obs["Inflammation_binary"] = mdata_stroma["rna"].obs["Inflammation_binary"].cat.reorder_categories(["Non_Inflamed", "Inflamed"])

In [ ]:
milo.da_nhoods(mdata_stroma, design="~Inflammation_score", solver="edger")

In [ ]:
milo.build_nhood_graph(mdata_stroma)

In [ ]:
plt.rcParams["figure.figsize"] = [7, 7]
milo.plot_nhood_graph(
    mdata_stroma,
    padj_threshold=0.1,  # SpatialFDR level (1%)
    min_size=1,  # Size of smallest dot
)

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='hgca_celltype_v1')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Inflammation_score')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Remission_status')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Patient')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='Treatment')

In [ ]:
sc.pl.umap(taurus_adata_list[lin_idx], color='final_analysis', legend_loc="on data", legend_fontsize=8)

### HGCA cell types

In [ ]:
milo.annotate_nhoods(mdata_stroma, anno_col="hgca_celltype_v1")

In [ ]:
milo.plot_da_beeswarm(mdata_stroma, padj_threshold=0.1)

### On author labels

In [ ]:
milo.annotate_nhoods(mdata_stroma, anno_col="final_analysis")

In [ ]:
milo.plot_da_beeswarm(mdata_stroma, padj_threshold=0.1)

In [ ]:
adata = lin_adata

In [ ]:
sc.pp.normalize_total(adata, target_sum=10e4)
sc.pp.log1p(adata)

In [ ]:
import numpy as np, pandas as pd, scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt, seaborn as sns

L1, L2 = "hgca_celltype_v1", "final_analysis"

# pseudobulk correlation assumes log-normalized values, NOT raw counts
print(adata.X.min(), adata.X.max(), adata.X.dtype)   # max should be ~5-10, not thousands

In [ ]:
X = adata.X.toarray() if sp.issparse(adata.X) else np.asarray(adata.X)
expr = pd.DataFrame(X, index=adata.obs_names, columns=adata.var_names)

def pseudobulk(expr, labels, min_cells=10):
    lab = pd.Series(np.asarray(labels).astype(str), index=expr.index)
    keep = lab.value_counts()[lambda s: s >= min_cells].index
    pb = expr[lab.isin(keep).values].groupby(lab[lab.isin(keep)].values).mean()
    return pb

pb1 = pseudobulk(expr, adata.obs[L1])   # clusters x genes
pb2 = pseudobulk(expr, adata.obs[L2])

# z-score each gene across the *union* of both label sets so the two
# matrices sit in the same space and highly-expressed genes don't dominate
both = pd.concat([pb1, pb2])
z = (both - both.mean(0)) / both.std(0).replace(0, np.nan)
z = z.dropna(axis=1)
z1, z2 = z.iloc[:len(pb1)], z.iloc[len(pb1):]

# Pearson correlation between every hgca cluster and every final_analysis cluster
sim = pd.DataFrame(np.corrcoef(z1.values, z2.values)[:len(z1), len(z1):],
                   index=z1.index, columns=z2.index)

g = sns.clustermap(sim, cmap="RdBu_r", center=0, figsize=(12, 10),
                   cbar_kws={"label": "Pearson r (pseudobulk, gene-z)"},
                   annot=False, linewidths=.2)
g.ax_heatmap.set_xlabel(L2); g.ax_heatmap.set_ylabel(L1)
plt.show()

## Check specific markers